In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append(r"C:\Users\Admin\Desktop\master_thesis\volume_prediction_fip")

import json
import cv2
import os
import pandas as pd
from fip_detection import detect

dir = r"F:\FIP-data\images\2023\WW034\debayered\2023_06_08_13_11_Lot1\FPWW0340179_FIP2_20230608_122932"
df = pd.read_csv(r"F:\FIP-data\csv\precomputed.csv")
data = json.loads(df.loc[df["image_dir"] == dir, "spikes"].iloc[0])
with open("../assets/poses/2023_06_08_13_11_Lot1_scaled.json") as f:
    conf = json.load(f)
boxes = {} # Ensure ids are unique
for img, boxes_img in data.items():
    n = {}
    for i, (_, v) in enumerate(boxes_img):
        n[i] = v
    boxes[img] = n

In [2]:
import numpy as np
np.set_printoptions(suppress=True, precision=6)
min_view = 6
np.random.seed(0)

In [6]:
# Testcode. Can be added to estimate_distance to export sampled points respectively result points
# Test
"""
valid_estimates = estimates[mask, :3]
color = np.random.uniform(0, 1, 3)
with open(r"F:\FIP-data\images\2023\WW034\debayered\2023_06_08_13_11_Lot1\FPWW0340108_FIP2_20230608_122109\out.obj", 'a') as f:
    for vertex in valid_estimates:
        f.write(f"v {vertex[0]} {vertex[1]} {vertex[2]} {color[0]} {color[1]} {color[2]}\n")

with open(r"F:\FIP-data\images\2023\WW034\debayered\2023_06_08_13_11_Lot1\FPWW0340108_FIP2_20230608_122109\out_mean.obj", 'a') as f:
    f.write(f"v {estimate[0]} {estimate[1]} {estimate[2]} {color[0]} {color[1]} {color[2]}\n")
"""

<>:3: SyntaxWarning: invalid escape sequence '\F'
<>:3: SyntaxWarning: invalid escape sequence '\F'
C:\Users\Admin\AppData\Local\Temp\ipykernel_28952\3998624096.py:3: SyntaxWarning: invalid escape sequence '\F'
  """


'\nvalid_estimates = estimates[mask, :3]\ncolor = np.random.uniform(0, 1, 3)\nwith open(r"F:\\FIP-data\\images\x823\\WW034\\debayered\x823_06_08_13_11_Lot1\\FPWW0340108_FIP2_20230608_122109\\out.obj", \'a\') as f:\n    for vertex in valid_estimates:\n        f.write(f"v {vertex[0]} {vertex[1]} {vertex[2]} {color[0]} {color[1]} {color[2]}\n")\n\nwith open(r"F:\\FIP-data\\images\x823\\WW034\\debayered\x823_06_08_13_11_Lot1\\FPWW0340108_FIP2_20230608_122109\\out_mean.obj", \'a\') as f:\n    f.write(f"v {estimate[0]} {estimate[1]} {estimate[2]} {color[0]} {color[1]} {color[2]}\n")\n'

In [3]:
combined_results = detect.connect_boxes(boxes, conf)

In [9]:
def save_results(combined_results, dir_in, dir_out, write_distance = True, write_id = True, shade_distance = False):
    import matplotlib.pyplot as plt

    dist_cmap = plt.get_cmap("jet")
    colors = {-1 : (1.0, 0.0, 0.0)}

    boxes = {}
    for b in combined_results:
        boxes.setdefault(b["image"], [])
        boxes[b["image"]].append((b["cluster"], b["box"], b["distance"]))

    # Iterate through all 12 result objects
    imgs = [f"cam_{i:02}.png" for i in range(1, 3)]
    for img_name in imgs:
        img = cv2.imread(os.path.join(dir_in, img_name))

        # Get the bounding boxes and IDs for the current image
        for id, bbox, distance in boxes[img_name]:
            x1, y1, x2, y2 = bbox  # Extract the coordinates of the bounding box
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            
            # Draw the bounding box
            if shade_distance:
                if distance:
                    n_dist = np.clip((distance - 2.5) / (3.5 - 2.5), 0, 1)
                    color = dist_cmap(n_dist)
                else:
                    color = (1.0, 0, 0)
            else:
                colors.setdefault(id, np.random.rand(3))
                color = colors[id]
            img = cv2.rectangle(img, (x1, y1), (x2, y2), color=(int(color[0] * 255), int(color[1] * 255), int(color[2] * 255)), thickness=2)
            
            # Annotate with the ID (inside the bounding box)
            if distance and write_distance and write_id:
                text = f"{id}, {distance:.2f}"
            elif distance and write_distance:
                text = f"{distance:.2f}"
            elif write_id:
                text = f"{id}"
            else:
                text = ""
            if write_id or write_distance:
                img = cv2.putText(img, text, (x1 + 5, y1 + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (int(color[0] * 255), int(color[1] * 255), int(color[2] * 255)), 2)

        out_path = os.path.join(dir_out, f"visu_{os.path.splitext(img_name)[0]}.jpg")
        cv2.imwrite(out_path, img, [cv2.IMWRITE_JPEG_QUALITY, 70])

save_results(combined_results, dir, os.path.join(dir, "test"), True, False, True)